In [21]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.interpolate import interp1d

import os
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import quad
from scipy.optimize import minimize 
from datetime import datetime as dt

from eod import EodHistoricalData
from nelson_siegel_svensson import NelsonSiegelSvenssonCurve
from nelson_siegel_svensson.calibrate import calibrate_nss_ols

In [34]:
option_data = pd.read_excel(r'C:\Users\S084339\OneDrive - Abeille Assurances\WARNA DRIVE\THESIS PRICING\OPTION_DATA_to_paste_in_final_datatable.xlsx', header = 0)
option_data = option_data[option_data['Type']=='Call']

option_data = option_data[option_data["Volume"]>0]
option_data = option_data[option_data["Moneyness S/K"]>0.85]
option_data = option_data[option_data["Moneyness S/K"]<1.15]
option_data = option_data[option_data["Mid"]>=10]

option_data = option_data[option_data["IV"]>0]
option_data = option_data[option_data["DaysToMaturity"]>7]
option_data = option_data[option_data["DaysToMaturity"]<365.25]


option_data = option_data[option_data["Mid"]<300]


options_data = option_data.copy()#.sample(n=800)
options_data

,Type,Strike,Spot,Expiration Date,Bid,Ask,Last Sale,Volume,IV,Delta,Gamma,Open Interest,Net,DaysToMaturity,Moneyness S/K,Mid,Rate
1003,Call,4775,5048.8198,Fri Apr 26 2024,278.4,284.2,267.19,1,0.2100,0.9590,0.0005,1,1.44,8,1.057344,281.30,TBD
1026,Call,4915,5048.8198,Fri Apr 26 2024,147.4,152.7,141.67,1,0.1814,0.8468,0.0017,1,4.27,8,1.027227,150.05,TBD
1028,Call,4925,5048.8198,Fri Apr 26 2024,139.2,144.4,116.82,10,0.1817,0.8312,0.0018,34,-12.68,8,1.025141,141.80,TBD
1031,Call,4940,5048.8198,Fri Apr 26 2024,126.3,131.3,123.36,2,0.1793,0.8052,0.0020,4,5.76,8,1.022028,128.80,TBD
1037,Call,4970,5048.8198,Fri Apr 26 2024,102.6,106.3,96.40,9,0.1723,0.7434,0.0024,43,1.70,8,1.015859,104.45,TBD
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3974,Call,5850,5048.8198,Fri Jan 17 2025,47.6,48.5,45.95,42,0.1297,0.1716,0.0004,178,-0.25,274,0.863046,48.05,TBD
3984,Call,5850,5048.8198,Fri Feb 21 2025,62.4,63.5,61.96,240,0.1316,0.2010,0.0005,218,1.51,309,0.863046,62.95,TBD
4006,Call,5900,5048.8198,Fri Mar 21 2025,65.9,67.0,62.95,1,0.1325,0.2019,0.0004,6122,-1.10,337,0.855732,66.45,TBD
4007,Call,5925,5048.8198,Fri Mar 21 2025,61.1,62.3,59.95,8,0.1318,0.1914,0.0004,278,0.10,337,0.852121,61.70,TBD


In [24]:
def heston_charfunc(phi, S0, v0, kappa, theta, sigma, rho, lambd, tau, r):
    
    # constants
    a = kappa*theta
    b = kappa+lambd
    
    # common terms w.r.t phi
    rspi = rho*sigma*phi*1j
    
    # define d parameter given phi and b
    d = np.sqrt( (rho*sigma*phi*1j - b)**2 + (phi*1j+phi**2)*sigma**2 )
    
    # define g parameter given phi, b and d
    g = (b-rspi+d)/(b-rspi-d)
    
    # calculate characteristic function by components
    exp1 = np.exp(r*phi*1j*tau)
    term2 = S0**(phi*1j) * ( (1-g*np.exp(d*tau))/(1-g) )**(-2*a/sigma**2)
    exp2 = np.exp(a*tau*(b-rspi+d)/sigma**2 + v0*(b-rspi+d)*( (1-np.exp(d*tau))/(1-g*np.exp(d*tau)) )/sigma**2)

    return exp1*term2*exp2

def integrand(phi, S0, v0, kappa, theta, sigma, rho, lambd, tau, r):
    try:
        args = (S0, v0, kappa, theta, sigma, rho, lambd, tau, r)
        numerator = np.exp(r*tau)*heston_charfunc(phi-1j,*args) - K*heston_charfunc(phi,*args)
        denominator = 1j*phi*K**(1j*phi)
        return numerator/denominator
    except OverflowError:
        return 0

def heston_price_rec(S0, K, v0, kappa, theta, sigma, rho, lambd, tau, r):
    args = (S0, v0, kappa, theta, sigma, rho, lambd, tau, r)
    
    P, umax, N = 0, 100, 10000
    dphi=umax/N #dphi is width

    for i in range(1,N):
        # rectangular integration
        phi = dphi * (2*i + 1)/2 # midpoint to calculate height
        numerator = np.exp(r*tau)*heston_charfunc(phi-1j,*args) - K * heston_charfunc(phi,*args)
        denominator = 1j*phi*K**(1j*phi)
        
        P += dphi * numerator/denominator
        
    return np.real((S0 - K*np.exp(-r*tau))/2 + P/np.pi)

def heston_price(S0, K, v0, kappa, theta, sigma, rho, lambd, tau, r):
    args = (S0, v0, kappa, theta, sigma, rho, lambd, tau, r)
    
    real_integral, err = np.real( quad(integrand, 0, 100, args=args) )
    
    return (S0 - K*np.exp(-r*tau))/2 + real_integral/np.pi

# Parameters to test model

S0 = 100. # initial asset price
K = 100. # strike
v0 = 0.2 # initial variance
r = 0.03 # risk free rate
kappa = 1.5768 # rate of mean reversion of variance process
theta = 0.0398 # long-term mean variance
sigma = 0.3 # volatility of volatility
lambd = 0.575 # risk premium of variance
rho = -0.5711 # correlation between variance and stock process
tau = 1. # time to maturity

print(heston_price( S0, K, v0, kappa, theta, sigma, rho, lambd, tau, r ))

print(heston_price_rec( S0, K, v0, kappa, theta, sigma, rho, lambd, tau, r ))

C:\Dex\application\python\Lib\site-packages\scipy\integrate\_quadpack_py.py:611: ComplexWarning: Casting complex values to real discards the imaginary part
  return _quadpack._qagse(func,a,b,args,full_output,epsabs,epsrel,limit)


14.430957796960113
14.398642884536581


In [25]:
# Extract yield curve and create an interpolation function
yield_maturities = np.array([1/12, 2/12, 3/12, 4/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30])
yields = np.array([5.49, 5.51, 5.45, 5.44, 5.39, 5.18, 4.97, 4.83, 4.69, 4.69, 4.67, 4.88, 4.77])/100
yield_curve = interp1d(yield_maturities, yields, kind='cubic', fill_value='extrapolate')

print(yield_curve)

In [37]:
from scipy.optimize import differential_evolution, minimize
import numpy as np
import pandas as pd

# Assuming options_data and yield_curve are already defined as shown in previous examples.

def calibration_error(params, data, yield_curve):
    kappa, sigma, rho = params
    # Fixed parameters
    v0 = 0.23
    theta = 0.23
    lambd = 0
    total_error = 0
    for _, row in data.iterrows():
        r = yield_curve(row['DaysToMaturity'])
        model_price = heston_price(row['Spot'], row['Strike'], v0, kappa, theta, sigma, rho, lambd, row['DaysToMaturity']/365.25, r)
        total_error += (model_price - row['Mid'])**2
    return total_error

# Define bounds for the parameters still being optimized
bounds = [(0.01, 3), (0.1, 1.0), (-1, 0)]


# Global Optimization using Differential Evolution
#global_result = differential_evolution(calibration_error, bounds, args=(options_data, yield_curve), strategy='best1bin', maxiter=1000, popsize=15, tol=0.01, mutation=(0.5, 1), recombination=0.7)
global_result = differential_evolution(calibration_error, bounds, args=(options_data, yield_curve), strategy='best1bin', maxiter=1000, popsize=15, tol=0.01, mutation=(0.5, 1), recombination=0.7)

print("Global Optimization Results:")
print("Success:", global_result.success)
print("Message:", global_result.message)
print("Optimized Parameters:", global_result.x)



C:\Users\S084339\AppData\Local\Temp\ipykernel_42148\2974181208.py:51: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  real_integral, err = np.real( quad(integrand, 0, 100, args=args) )


Global Optimization Results:
Success: True
Message: Optimization terminated successfully.
Optimized Parameters: [ 0.90630302  0.93611521 -0.5586409 ]


1) Optimized Parameters: kappa, theta, sigma, rho, lambd, v0 = [ 1.97590419  0.28603668  0.87293317 -0.51169262  0.38240857  0.18189543]
2) Optimized parameters : kappa sigma rho =  [ 1.94672706  0.68071191 -0.53282122] with v0 = 0.23, theta = 0.23, lambd = 0
3) Optimized parameters : kappa sigma rho = [ 0.91546582  0.26325824 -0.52765504] with v0 = 0.23, theta = 0.23, lambd = 0 with more data
4) Optimized parameters : kappa sigma rho = [ 2.07673065  0.37993998 -0.70559848] with v0 = 0.23, theta = 0.23, lambd = 0 with EVEN more data
5) Optimized parameters : kappa sigma rho = [ 0.49846034  0.56942014 -0.56090413] with v0 = 0.23, theta = 0.23, lambd = 0 with EVEN more data
6) Optimized parameters : kappa sigma rho = [ 1.38862568  0.64468182 -0.94955944] with v0 = 0.23, theta = 0.23, lambd = 0 with EVEN more data
7) Optimized parameters : kappa sigma rho = [ 2.32144983  0.14868047 -0.84942791] with v0 = 0.23, theta = 0.23, lambd = 0 with EVEN more data
8) Optimized parameters : kappa sigma rho = [ 0.90630302  0.93611521 -0.5586409 ] with v0 = 0.23, theta = 0.23, lambd = 0 with ALL data and 1000

## ESSAYER SUR HESTON DANS L'AUTRE PYTHON CELUI QUI A LA PLUS FAIBLE RMSE

In [38]:
# Use results from global optimization as initial guesses for local optimization
initial_params = global_result.x

#initial_params = [ 1.61930098,  0.25057689,  0.3, -0.7, 0,  0.23340976]

# Local Optimization using BFGS for more precise calibration
local_result = minimize(calibration_error, initial_params, args=(options_data, yield_curve), method='L-BFGS-B', bounds=bounds)

print("\nLocal Optimization Results:")
print("Success:", local_result.success)
print("Message:", local_result.message)
print("Optimized Parameters:", local_result.x)


C:\Users\S084339\AppData\Local\Temp\ipykernel_42148\2974181208.py:51: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  real_integral, err = np.real( quad(integrand, 0, 100, args=args) )



Local Optimization Results:
Success: True
Message: CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Optimized Parameters: [ 0.90630302  0.93611521 -0.5586409 ]
